In [43]:
import torch
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import torch.optim as optim
import json
from tqdm.notebook import tqdm
import gc

In [44]:
gc.collect()
# Clear PyTorch's resident memory
torch.cuda.empty_cache()

Face verification data loading for AUC and ROC calculation

In [45]:
class FaceVerificationDataset(torch.utils.data.Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.pairs = []

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 3:
                    self.pairs.append((parts[0], parts[1], int(parts[2])))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]

        img1_full_path = os.path.join(self.root_dir, img1_path)
        img2_full_path = os.path.join(self.root_dir, img2_path)

        img1 = Image.open(img1_full_path).convert('RGB')
        img2 = Image.open(img2_full_path).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

Data loading and augmentation:

In [46]:

transformation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [47]:
train_dataset = ImageFolder(root='../data/classification_data/train_data', transform=transformation)
train_loader = DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=True,    
    num_workers=4,   
    pin_memory=True   
)

In [48]:
class_val_dataset = ImageFolder(
    root='../data/classification_data/val_data', 
    transform=transformation,

)
class_val_loader = DataLoader(
    class_val_dataset, 
    batch_size=64, 
    shuffle=False, 
    num_workers=4
)

verfication loading

In [49]:
verification_root = '../data' 

verification_dataset = FaceVerificationDataset(
    txt_file='../data/verification_pairs_val.txt', 
    root_dir=verification_root, 
    transform= transformation
)

verification_loader = torch.utils.data.DataLoader(verification_dataset, batch_size=32, shuffle=False)

Calculating ROC and AUC

In [50]:
def validate_verification_auc(model, val_loader, device):
    model.eval()
    all_labels = []
    all_cosine_scores = []
    all_euclidean_distances = []

    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2 = img1.to(device), img2.to(device)

            feat1 = model.features(img1)
            feat1 = model.avgpool(feat1).flatten(1)
            
            feat2 = model.features(img2)
            feat2 = model.avgpool(feat2).flatten(1)

            cos_sim = F.cosine_similarity(feat1, feat2)
            euc_dist = torch.cdist(feat1.unsqueeze(1), feat2.unsqueeze(1)).squeeze()
            
            all_cosine_scores.extend(cos_sim.cpu().numpy())
            all_euclidean_distances.extend(euc_dist.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            

    auc_cosine = roc_auc_score(all_labels, all_cosine_scores)
    auc_euclidean = roc_auc_score(all_labels, -np.array(all_euclidean_distances))

    return auc_cosine, auc_euclidean

In [51]:

class CustomCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(CustomCNN, self).__init__()
        
        # 1. Convolutional Backbone (Feature Extractor)
        self.features = nn.Sequential(
            # Conv Block 1: Input channels=3 (RGB), Output channels=32
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 2: Input=32, Output=64
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 3: Input=64, Output=128
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # 2. Global Average Pooling 
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(in_features=128, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=128, out_features=num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, start_dim=1)  # Flattens the 1x1 spatial grid to an array of 128
        x = self.classifier(x)
        return x

In [52]:
num_classes = len(train_dataset.classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = CustomCNN(num_classes)

model = model.to(device)
model.eval()

Using device: cuda


CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=4000, bias=True)
  )
)

In [53]:
for param in model.parameters():
    param.requires_grad = True

optimizer =optim.Adam([
    {'params': model.parameters(), 'lr': 1e-4}])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()

In [54]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device Count: {torch.cuda.device_count()}")

Is CUDA available? True
CUDA version: 13.0
Device Count: 1


In [ ]:
num_epochs = 30
best_auc = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss':[],
    'val_acc':[],
    'cos_auc': [],
    'euc_auc': []
}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)

    # --- TRAINING PHASE ---
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_preds += labels.size(0)
        correct_preds += (predicted == labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=correct_preds/total_preds)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in class_val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Calculate loss and accuracy
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_preds / total_preds
    epoch_val_loss = val_loss / len(class_val_dataset)
    epoch_val_acc = val_correct / val_total


    cos_auc, euc_auc = validate_verification_auc(model, verification_loader, device)

    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['cos_auc'].append(cos_auc)
    history['euc_auc'].append(euc_auc)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Train Acc: {epoch_val_acc:.4f}")
    print(f"Cosine AUC: {cos_auc:.4f} | Euclidean AUC: {euc_auc:.4f}")
    print("-" * 30)

    if cos_auc > best_auc:
        best_auc = cos_auc
        torch.save(model.state_dict(), 'best_supervised_model_cnn.pth')
        print("Model saved based on Cosine AUC!")

    scheduler.step(cos_auc)


# Save history to a JSON file
with open('training_history_supervised_cnn.json', 'w+') as f:
    json.dump(history, f)

Epoch [1/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [1/30]
Train Loss: 8.2213 | Train Acc: 0.0008
Val Loss: 8.2644 | Train Acc: 0.0006
Cosine AUC: 0.6281 | Euclidean AUC: 0.6342
------------------------------
Model saved based on Cosine AUC!


Epoch [2/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [2/30]
Train Loss: 8.0929 | Train Acc: 0.0015
Val Loss: 8.1471 | Train Acc: 0.0006
Cosine AUC: 0.6110 | Euclidean AUC: 0.6411
------------------------------


Epoch [3/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

...

Epoch [28/30]
Train Loss: 7.6929 | Train Acc: 0.0039
Val Loss: 7.8117 | Train Acc: 0.0036
Cosine AUC: 0.6028 | Euclidean AUC: 0.6413
------------------------------


Epoch [29/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [29/30]
Train Loss: 7.6915 | Train Acc: 0.0041
Val Loss: 7.8120 | Train Acc: 0.0035
Cosine AUC: 0.6019 | Euclidean AUC: 0.6407
------------------------------


Epoch [30/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [30/30]
Train Loss: 7.6925 | Train Acc: 0.0041
Val Loss: 7.8106 | Train Acc: 0.0037
Cosine AUC: 0.6024 | Euclidean AUC: 0.6412
------------------------------
